# Telemetry Playbook

This notebook explores analyst interaction telemetry captured in `frontend_observation_events`. Adjust the connection cell to point at your analytics replica before running these queries.

In [ ]:
from __future__ import annotations

import os
from datetime import datetime, timedelta

import pandas as pd
from sqlalchemy import create_engine, text

from cerebro.core.config import settings

In [ ]:
DATABASE_URL = os.environ.get("CEREBRO_ANALYTICS_DB_URL", settings.database_url)
engine = create_engine(DATABASE_URL)

In [ ]:
window_days = int(os.environ.get("TELEMETRY_WINDOW_DAYS", 7))
query = text(
    """
    SELECT
        occurred_at,
        event_type,
        component,
        agent_session_id,
        context_data
    FROM frontend_observation_events
    WHERE occurred_at >= now() - (:window * interval '1 day')
    ORDER BY occurred_at DESC
    """
)
observations = pd.read_sql_query(
    query.bindparams(window=window_days),
    engine,
    parse_dates=['occurred_at'],
)
observations.head()

In [ ]:
if observations.empty:
    raise RuntimeError("No telemetry rows returned; adjust window or data source.")

hourly = (
    observations
    .set_index('occurred_at')
    .groupby('event_type')
    .resample('1H')
    .size()
    .rename('count')
    .reset_index()
)
hourly.tail()

In [ ]:
daily_component = (
    observations
    .set_index('occurred_at')
    .groupby('component')
    .resample('1D')
    .size()
    .rename('daily_events')
    .reset_index()
)
pivot = daily_component.pivot(index='occurred_at', columns='component', values='daily_events').fillna(0)
pivot.tail()

In [ ]:
event_totals = observations.groupby('event_type').size().sort_values(ascending=False)
event_totals.head(10)